### 导入所需的库和模块

In [1]:
import psutil
import h5py
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Activation, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

### 定义一个函数来探索 HDF5 文件的内容和结构

In [ ]:
def explore_h5_file(file_path):
    """
    探索 HDF5 文件的内容和结构
    
    参数:
    file_path (str): H5 文件的路径
    """
    # 打开 H5 文件
    with h5py.File(file_path, 'r') as f:
        # 递归函数用于探索组和数据集
        def explore_group(name, obj):
            if isinstance(obj, h5py.Group):
                print(f"组: {name}")
                print(f"  属性: {dict(obj.attrs)}")
                print(f"  包含 {len(obj.items())} 个项目")
            elif isinstance(obj, h5py.Dataset):
                print(f"数据集: {name}")
                print(f"  形状: {obj.shape}")
                print(f"  类型: {obj.dtype}")
                print(f"  属性: {dict(obj.attrs)}")
                
                # 尝试显示数据集的部分内容
                try:
                    if len(obj.shape) > 0:  # 确保不是标量
                        # 对于大型数据集，只显示前几个元素
                        if obj.shape[0] > 5:
                            print(f"  前5个值: {obj[:5]}")
                        else:
                            print(f"  值: {obj[:]}")
                    else:
                        print(f"  值: {obj[()]}")
                except Exception as e:
                    print(f"  无法显示值: {e}")
        
        # 列出文件中的所有组和数据集
        print("H5 文件结构:")
        f.visit(lambda name: print(f"- {name}"))
        
        print("\n详细信息:")
        # 遍历所有项目并显示详细信息
        f.visititems(explore_group)

# 使用示例
if __name__ == "__main__":
    file_path = "ign.h5"  # 替换为你的 HDF5 文件路径
    explore_h5_file(file_path)


### 定义模型架构并加载权重

In [ ]:
# 步骤1：定义模型架构并加载权重
model = Sequential([
    Conv2D(24, (16, 1), padding='valid', input_shape=(24, 3, 1)),
    Activation('relu'),
    MaxPooling2D(pool_size=(3, 1)),
    Flatten(),
    Dense(12, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')
])
model.load_weights('ign.h5')  # 加载现有权重
model.summary()

### 编译模型

In [4]:
# 步骤2：编译模型
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

### 加载和预处理WISDM数据集

In [ ]:
# 步骤3：加载和预处理WISDM数据集
# 检查内存使用
def check_memory():
    mem = psutil.virtual_memory()
    print(f"内存使用率: {mem.percent}% 可用: {mem.available / (1024 ** 3):.2f} GB")

def load_wisdm_data(file_path, time_steps=24, max_samples=None):
    """
    加载和预处理WISDM数据集，确保数值类型并限制样本数以避免内存问题。
    
    Args:
        file_path (str): 数据文件路径。
        time_steps (int): 时间步长。
        max_samples (int): 最大样本数（可选，防止内存溢出）。
    
    Returns:
        tuple: (data, labels)。
    """
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            fields = line.strip().split(',')[:6]
            if len(fields) == 6:
                data.append(fields)
    
    df = pd.DataFrame(data, columns=['user_id', 'activity', 'timestamp', 'x', 'y', 'z'])
    
    # 转换为浮点数
    df['x'] = pd.to_numeric(df['x'], errors='coerce')
    df['y'] = pd.to_numeric(df['y'], errors='coerce')
    df['z'] = pd.to_numeric(df['z'], errors='coerce')
    df = df.dropna()
    
    # 映射活动标签（适配WISDM数据集的实际活动）
    activity_map = {
        'Jogging': 0,      # 慢跑
        'Sitting': 1,      # 坐着（归类为“静止”）
        'Standing': 1,     # 站立（归类为“静止”）
        'Upstairs': 2,     # 上楼梯（归类为“楼梯”）
        'Downstairs': 2,   # 下楼梯（归类为“楼梯”）
        'Walking': 3       # 步行
    }
    df['activity'] = df['activity'].map(activity_map)
    df = df.dropna(subset=['activity'])  # 删除未映射的活动
    
    print("DataFrame dtypes:\n", df.dtypes)
    print(f"数据行数: {len(df)}")
    check_memory()
    
    data_list = []
    labels_list = []
    
    for (user_id, activity), group in df.groupby(['user_id', 'activity']):
        group = group.sort_values('timestamp')
        features = group[['x', 'y', 'z']].values
        for i in range(0, len(features) - time_steps + 1, time_steps):
            sample = features[i:i + time_steps]
            if sample.shape[0] == time_steps:
                data_list.append(sample.reshape(time_steps, 3, 1))
                labels_list.append(activity)
                if max_samples and len(data_list) >= max_samples:
                    break
        if max_samples and len(data_list) >= max_samples:
            break
    
    data = np.array(data_list, dtype=np.float32)
    labels = np.array(labels_list)
    print(f"生成样本数: {len(data)}")
    check_memory()
    
    return data, labels

# 数据路径
dataset_path = 'dataset/WISDM_ar_v1.1_raw.txt'

# 加载数据（限制样本数以防内存溢出）
data, labels = load_wisdm_data(dataset_path, time_steps=24, max_samples=10000)



### 分割数据为训练集和验证集

In [6]:
# 步骤4：分割数据为训练集和验证集
validation_split = 0.2
train_data, val_data, train_labels, val_labels = train_test_split(
    data, labels, test_size=validation_split, random_state=123, stratify=labels
)

### 对标签进行独热编码

In [ ]:
# 步骤5：对标签进行独热编码
encoder = OneHotEncoder(sparse_output=False, categories=[[0, 1, 2, 3]])
train_labels = encoder.fit_transform(train_labels.reshape(-1, 1))
val_labels = encoder.transform(val_labels.reshape(-1, 1))

# 验证数据形状
print(f"训练数据形状: {train_data.shape}, 训练标签形状: {train_labels.shape}")
print(f"验证数据形状: {val_data.shape}, 验证标签形状: {val_labels.shape}")

### 设置回调函数

In [8]:
# 步骤6：设置回调函数
reduce_lr = ReduceLROnPlateau(monitor='val_accuracy', mode='max', factor=0.5, patience=40, min_lr=1e-5)
early_stopping = EarlyStopping(monitor='val_accuracy', mode='max', restore_best_weights=True, patience=60)
callbacks = [reduce_lr, early_stopping]

### 训练模型

In [ ]:
# 步骤7：训练模型
history = model.fit(
    train_data,
    train_labels,
    validation_data=(val_data, val_labels),
    batch_size=256,
    epochs=200,
    callbacks=callbacks
)

### 保存新训练的模型（可选）

In [ ]:
# 步骤8：保存新训练的模型（可选）
model.save('newly_trained_model.h5')  # 保存整个模型